# NB-03 · End-to-End Evaluation
Generates: main results figures, N-sensitivity, multiseed stability, complementarity.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath('.'), '.'))
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from utils import (
    setup_style, save_fig, load_results_history, load_multiseed,
    PALETTE, STRATEGY_ORDER, STRATEGY_LABELS, FIG_DIR
)

setup_style()

history = load_results_history()
multiseed = load_multiseed()
print(f'Loaded {len(history)} runs, multiseed n_seeds={multiseed["n_seeds"]}')

## 1. Canonical Numbers (sourced from evaluation JSON + logs)

In [ ]:
# Canonical metrics (100k users, N=99, seed=42)
# Sourced from:
#   - run 20260426_233351  → Pipeline A, DIF-SASRec, System (A∪B)
#   - run 20260426_170513  → Content Baseline, GRU-SeqDQN
# HR@5 / MRR@10 for System (A∪B) not captured in joint eval (tracking limitation)

canonical = {
    'Random Baseline':      dict(hr5=0.050,  hr10=0.100,  ndcg10=0.0454, mrr10=None),
    'Content Baseline':     dict(hr5=0.3597, hr10=0.4346, ndcg10=0.3022, mrr10=0.2609),
    'GRU-SeqDQN':           dict(hr5=0.0519, hr10=0.1031, ndcg10=0.0472, mrr10=0.0306),
    'DIF-SASRec':           dict(hr5=0.5877, hr10=0.7745, ndcg10=0.5024, mrr10=0.4191),
    'Pipeline A (Cleora)':  dict(hr5=0.5750, hr10=0.9047, ndcg10=0.5393, mrr10=0.4302),
    'System (A∪B)':         dict(hr5=None,   hr10=0.9736, ndcg10=0.5571, mrr10=None),
}

for name, m in canonical.items():
    print(f'{name:30s}  HR@10={m["hr10"]:.4f}  NDCG@10={m["ndcg10"]:.4f}')

## 2. Figure: Main Results Bar Chart

In [ ]:
models = STRATEGY_ORDER
hr10   = [canonical[m]['hr10']   for m in models]
ndcg10 = [canonical[m]['ndcg10'] for m in models]

x = np.arange(len(models))
w = 0.38

fig, ax = plt.subplots(figsize=(9, 4.5))

colors_hr   = [PALETTE[m] for m in models]
colors_ndcg = [c + 'AA' for c in colors_hr]  # add transparency via alpha in patches

bars_hr   = ax.bar(x - w/2, hr10,   w, label='HR@10',   color=colors_hr,   edgecolor='white', linewidth=0.6)
bars_ndcg = ax.bar(x + w/2, ndcg10, w, label='NDCG@10', color=colors_hr,   edgecolor='white', linewidth=0.6, alpha=0.65, hatch='//')

# Annotate bars
for bar, val in zip(bars_hr, hr10):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars_ndcg, ndcg10):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([STRATEGY_LABELS[m] for m in models], fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.08)
ax.set_title('Main evaluation results (N=99 negatives, 100k users, seed=42)')

solid_patch = mpatches.Patch(color='#555', label='HR@10')
hatch_patch = mpatches.Patch(facecolor='#555', alpha=0.65, hatch='//', label='NDCG@10')
ax.legend(handles=[solid_patch, hatch_patch], loc='upper left')

# Highlight the final system
ax.axvline(x=len(models)-1.5, color='#555', linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(len(models)-1.45, 1.04, 'Combined system →', fontsize=8, color='#555', style='italic')

save_fig('fig_main_results', fig)
plt.show()

## 3. Figure: N-Sensitivity (N=99 vs N=999)

In [ ]:
# N=99 and N=999 HR@10 for 4 strategies (System A∪B not evaluated under N=999)
strategies_n = ['Content Baseline', 'GRU-SeqDQN', 'DIF-SASRec', 'Pipeline A (Cleora)']

n99_hr10  = [0.4346, 0.1031, 0.7745, 0.9047]
n999_hr10 = [0.2122, 0.0141, 0.3142, 0.3272]

x = np.arange(len(strategies_n))
w = 0.38

fig, ax = plt.subplots(figsize=(8, 4))
colors = [PALETTE[s] for s in strategies_n]

b1 = ax.bar(x - w/2, n99_hr10,  w, label='N = 99',  color=colors, edgecolor='white')
b2 = ax.bar(x + w/2, n999_hr10, w, label='N = 999', color=colors, edgecolor='white', alpha=0.55, hatch='..')

for bar, val in zip(b1, n99_hr10):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
for bar, val in zip(b2, n999_hr10):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([STRATEGY_LABELS[s] for s in strategies_n], fontsize=9)
ax.set_ylabel('HR@10')
ax.set_ylim(0, 1.08)
ax.set_title('HR@10 vs. Negative Sample Size (100k users)')
ax.legend()

# Random baseline lines
ax.axhline(0.100, color='#9E9E9E', linestyle='--', linewidth=0.9, label='Random N=99')
ax.axhline(0.010, color='#9E9E9E', linestyle=':', linewidth=0.9, label='Random N=999')

save_fig('fig_n_sensitivity', fig)
plt.show()

## 4. Figure: Multiseed Stability (DIF-SASRec)

In [ ]:
seeds     = multiseed['seeds']
hr10_vals = [s['hr10'] for s in multiseed['per_seed']]
ndcg_vals = [s['ndcg10'] for s in multiseed['per_seed']]
mean_hr   = multiseed['mean_hr10']
std_hr    = multiseed['std_hr10']
mean_nd   = multiseed['mean_ndcg10']
std_nd    = multiseed['std_ndcg10']

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, vals, mean, std, metric, color in [
    (axes[0], hr10_vals, mean_hr, std_hr,  'HR@10',   '#61AFEF'),
    (axes[1], ndcg_vals, mean_nd, std_nd,  'NDCG@10', '#98C379'),
]:
    bars = ax.bar(range(len(seeds)), vals, color=color, edgecolor='white', alpha=0.85)
    ax.axhline(mean, color='#E06C75', linewidth=1.5, linestyle='--', label=f'Mean={mean:.4f}')
    ax.fill_between([-0.5, len(seeds)-0.5], mean - std, mean + std,
                    color='#E06C75', alpha=0.15, label=f'±σ={std:.4f}')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(range(len(seeds)))
    ax.set_xticklabels([f'seed={s}' for s in seeds], rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(metric)
    lo = min(vals) - 0.01
    hi = max(vals) + 0.015
    ax.set_ylim(lo, hi)
    ax.set_title(f'DIF-SASRec {metric} across seeds')
    ax.legend(fontsize=9)

save_fig('fig_multiseed_stability', fig)
plt.show()

## 5. Figure: Pipeline Complementarity

In [ ]:
# Complementarity counts from canonical run log (20260426_233351)
# INFO  Complementarity  A∩B=70,557  A-only=19,909  B-only=6,893  neither=2,641
comp = {
    'A∩B (both hit)':    70557,
    'A only':            19909,
    'B only':             6893,
    'Neither':            2641,
}
total = sum(comp.values())  # 100,000
labels  = list(comp.keys())
values  = list(comp.values())
pcts    = [v / total * 100 for v in values]
colors  = ['#C678DD', '#98C379', '#61AFEF', '#9E9E9E']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Donut chart
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    values, labels=None, colors=colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
)
for at in autotexts:
    at.set_fontsize(10)
ax.legend(wedges, [f'{l} ({v:,})' for l, v in zip(labels, values)],
          loc='lower center', bbox_to_anchor=(0.5, -0.12), fontsize=9, ncol=2)
ax.set_title('(a) Hit distribution across pipelines\n(100k users, N=99)')

# Bar chart with counts
ax2 = axes[1]
bars = ax2.barh(range(len(labels)), values, color=colors, edgecolor='white')
ax2.set_yticks(range(len(labels)))
ax2.set_yticklabels(labels)
ax2.set_xlabel('Number of users')
ax2.set_title('(b) Absolute counts')
for bar, v, p in zip(bars, values, pcts):
    ax2.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
             f'{v:,} ({p:.1f}%)', va='center', fontsize=9)
ax2.set_xlim(0, max(values) * 1.3)

save_fig('fig_complementarity', fig)
plt.show()

print(f'Total users evaluated: {total:,}')
print(f'Coverage: {(total - comp["Neither"])/total*100:.1f}% (at least one pipeline hit)')

## 6. Summary Table

In [ ]:
print('=== Final Results Summary ===')
print(f'{"Model":30s} {"HR@5":>8} {"HR@10":>8} {"NDCG@10":>10} {"MRR@10":>10}')
print('-' * 72)
for m in STRATEGY_ORDER:
    c = canonical[m]
    def fmt(v): return f'{v:.4f}' if v is not None else '  —   '
    print(f'{m:30s} {fmt(c["hr5"]):>8} {fmt(c["hr10"]):>8} {fmt(c["ndcg10"]):>10} {fmt(c["mrr10"]):>10}')

print(f'\nDIF-SASRec multiseed: HR@10 = {mean_hr:.4f} ± {std_hr:.4f} (n=5 seeds, 100k users each)')